# Fin AI SLM — Bank Statement Narration → JSON (Colab smoke test)
Smoke test of the full flow: **A** synthetic data → **A2** humanize rewrite → **B** QLoRA SFT (1 step) → **B2** optional DPO (1 step) → **C** eval → **D** inference → **E** vLLM serving (GPU).
Input = **bank statement narration lines only** (e.g. `UPI/P2M/...`, `NACH-DR-...`, `ACH RTN CHG`, `REV-IMPS`, `ATM WDL`, NEFT/IMPS/RTGS). No SMS alerts.
This is **not** a real fine-tune; it only proves each component's code runs. No accuracy claims here.
Runtime: Colab → Runtime → Change runtime type → **T4 GPU**.

## 0. Setup
Installs run only on Colab.
**Are we using vLLM / Unsloth?**
- **Training default = PEFT + TRL** (BitsAndBytes 4-bit + LoRA + `SFTTrainer`).
- **Unsloth = real option:** `USE_UNSLOTH = True` tries Unsloth `FastLanguageModel` for faster, lower-memory QLoRA on a Colab T4. If the install or import fails (or there is no GPU), Component B **falls back to PEFT + TRL automatically** and prints `TRAIN_BACKEND=peft`.
- **vLLM = used for serving in Component E** (`USE_VLLM=True` default): installed only on Colab+GPU in that cell (not here in setup), loads the merged smoke model (or base if training skipped) and runs narration→JSON. Falls back to Transformers / regex on failure; skipped cleanly with `SKIP_GPU=1`.

In [ ]:
import os, sys, json, random, re
IN_COLAB = 'google.colab' in sys.modules
SKIP_GPU = os.environ.get('SKIP_GPU','0')=='1'
if IN_COLAB:
    !pip -q install transformers peft trl bitsandbytes datasets accelerate jsonschema
USE_UNSLOTH = True   # flip to False to force the PEFT+TRL path
UNSLOTH_INSTALLED = False
if IN_COLAB and USE_UNSLOTH:
    # Install may be flaky; never fail setup. Import is attempted later in Component B.
    try:
        !pip -q install "unsloth"
        UNSLOTH_INSTALLED = True
    except Exception as e:
        print('unsloth install failed -> will use PEFT+TRL:', repr(e)[:200])
BASE = '/content/slm_smoke' if IN_COLAB else os.environ.get('SMOKE_DIR','/workspace/deliverables/slm_colab_smoke')
DATA = f'{BASE}/data'; os.makedirs(DATA, exist_ok=True)
try:
    import torch; HAS_GPU = torch.cuda.is_available() and not SKIP_GPU
    print('GPU:', torch.cuda.get_device_name(0) if HAS_GPU else 'none')
except ImportError:
    HAS_GPU = False; print('torch not installed -> GPU components will be skipped')
STATUS = {}
print('IN_COLAB', IN_COLAB, '| BASE', BASE, '| USE_UNSLOTH', USE_UNSLOTH)

## Schema
Every narration maps to these 10 fields. `jsonschema` checks types/enums; field accuracy is checked separately.

In [ ]:
SCHEMA = {"type":"object","additionalProperties":False,
 "required":["category","sub_category","counterparty_name","counterparty_type","is_emi","lender_name",
             "is_bounce_or_return","is_reversal","p2p_or_p2m","confidence"],
 "properties":{
  "category":{"type":"string"},"sub_category":{"type":"string"},
  "counterparty_name":{"type":["string","null"]},
  "counterparty_type":{"enum":["merchant","individual","lender","employer","bank","unknown"]},
  "is_emi":{"type":"boolean"},"lender_name":{"type":["string","null"]},
  "is_bounce_or_return":{"type":"boolean"},"is_reversal":{"type":"boolean"},
  "p2p_or_p2m":{"enum":["P2P","P2M","NA"]},
  "confidence":{"type":"number","minimum":0,"maximum":1}}}
FIELDS = list(SCHEMA['required'])

## Component A — Synthetic data (statement narrations)
**What:** template generator producing cryptic bank-statement narrations, labelled *by construction* (we know the label because we built the line).
**Why:** no real customer data needed; we control coverage of tricky cases (EMI, bounce, reversal).
Splits: `train`, `val` (same templates), `ood` (held-out templates — tests generalisation).

In [ ]:
random.seed(7)
MER=['SWIGGY','ZOMATO','AMAZON PAY','FLIPKART','BIGBASKET','IRCTC','JIO PREPAID']
PPL=['RAHUL SHARMA','PRIYA N','ANIL KUMAR','SNEHA R','MOHD IRFAN']
LEN=[('BAJAJFIN','Bajaj Finance'),('HDFCBANK LN','HDFC Bank'),('TATACAP','Tata Capital'),('IDFCFIRST','IDFC First Bank')]
EMP=['INFOSYS LTD','TCS LTD','ACME PVT LTD']
r=lambda n:''.join(random.choice('0123456789') for _ in range(n))
def lab(cat,sub,cp,cpt,emi=False,ln=None,bnc=False,rev=False,pp='NA'):
    return dict(category=cat,sub_category=sub,counterparty_name=cp,counterparty_type=cpt,is_emi=emi,lender_name=ln,
                is_bounce_or_return=bnc,is_reversal=rev,p2p_or_p2m=pp,confidence=1.0)
def t_upi_p2m():
    m=random.choice(MER); return f'UPI/P2M/{r(12)}/{m}/PAYMENT FROM PHONE/YESB', lab('expense','merchant_payment',m,'merchant',pp='P2M')
def t_upi_p2p():
    p=random.choice(PPL); return f'UPI/P2A/{r(12)}/{p}/UPI/SBIN', lab('transfer','p2p_transfer',p,'individual',pp='P2P')
def t_nach():
    c,n=random.choice(LEN); return f'NACH-DR-{c}-{r(10)}', lab('loan','emi_payment',n,'lender',emi=True,ln=n)
def t_achrtn():
    c,n=random.choice(LEN); return f'ACH RTN CHG {c} {r(8)}', lab('charges','emi_bounce_charge',n,'lender',emi=True,ln=n,bnc=True)
def t_rev():
    return f'REV-IMPS-{r(12)}-FAILED TXN', lab('refund','imps_reversal',None,'bank',rev=True)
def t_atm():
    return f'ATM WDL/{r(6)}/MUMBAI ANDHERI', lab('cash','atm_withdrawal',None,'bank')
def t_sal():
    e=random.choice(EMP); return f'NEFT CR-HDFC0000{r(3)}-{e}-SALARY {random.choice(["JAN","FEB","MAR"])}', lab('income','salary',e,'employer')
# OOD templates (different dialects, held out from train)
def o_rtgs():
    e=random.choice(EMP); return f'RTGS/UTIB{r(10)}/{e}/SAL', lab('income','salary',e,'employer')
def o_ecs():
    c,n=random.choice(LEN); return f'ECS/{c}/LOAN EMI {r(6)}', lab('loan','emi_payment',n,'lender',emi=True,ln=n)
def o_imps():
    p=random.choice(PPL); return f'MMT/IMPS/{r(12)}/{p}/KKBK', lab('transfer','p2p_transfer',p,'individual',pp='P2P')
IN=[t_upi_p2m,t_upi_p2p,t_nach,t_achrtn,t_rev,t_atm,t_sal]; OOD=[o_rtgs,o_ecs,o_imps]
mk=lambda fs,n:[dict(zip(('text','label'),random.choice(fs)())) for _ in range(n)]
splits={'train':mk(IN,70),'val':mk(IN,15),'ood':mk(OOD,15)}
for k,v in splits.items():
    with open(f'{DATA}/{k}.jsonl','w') as f: f.writelines(json.dumps(x)+'\n' for x in v)
import jsonschema
assert all(jsonschema.Draft7Validator(SCHEMA).is_valid(x['label']) for v in splits.values() for x in v)
print({k:len(v) for k,v in splits.items()}); print(json.dumps(splits['train'][0],indent=1))
STATUS['A_data']='PASS'

## Component A2 — Humanize / dialect rewrite (offline, deterministic)
**What:** rewrite ~30% of template lines so they look like messy real statement narrations: truncation, delimiter swaps, dropped segments, case changes, NACH/ECS/e-mandate dialect variants, OCR-ish char swaps on digits/refs, truncated merchant names, reordered UPI segments, trailing ref noise.
**Label safety:** labels are copied unchanged, then **consistency rules** check the rewritten text still carries the cues the label depends on (e.g. `is_emi` → lender cue still present; `is_bounce_or_return` → RTN/RETURN cue; `is_reversal` → REV cue; `counterparty_name` prefix still visible). If a rule fails, the rewrite is **dropped** and the clean line kept (counted in `dropped`).
**OOD:** some dialect variants (`ECS DR`, `MANDATE`, `ACH-D`) are only used on the OOD split, so eval sees unseen dialects.
Output: `*_human.jsonl` beside the clean `*.jsonl`. A teacher-LLM rewrite stub (not called) follows.

In [ ]:
HUMAN_FRAC=0.30; hrng=random.Random(11)
DIALECT_IN ={'NACH-DR':['NACH DR','NACH/DR','NACHDR']}
DIALECT_OOD={'NACH-DR':['ECS DR','MANDATE DR','ACH-D','E-MANDATE']}
OCR={'0':'O','1':'I','5':'S','8':'B','2':'Z'}
def aug_delim(t,R): d=R.choice([' ','-','|',' / ']); return re.sub(r'[/\-]',d,t)
def aug_case(t,R): return R.choice([t.lower(),t.upper(),t.title()])
def aug_ocr(t,R):
    def f(m): s=list(m.group()); i=R.randrange(len(s)); s[i]=OCR.get(s[i],s[i]); return ''.join(s)
    return re.sub(r'\d{6,}',f,t,count=1)
def aug_trunc_tail(t,R): return t[:max(12,int(len(t)*R.uniform(0.6,0.9)))]
def aug_drop_seg(t,R):
    p=t.split('/');
    if len(p)>3: p.pop(R.choice([i for i in range(len(p)) if i>=4] or [len(p)-1]))
    return '/'.join(p)
def aug_trunc_name(t,R):
    for n in MER+PPL+EMP:
        if n in t: return t.replace(n,n[:max(4,int(len(n)*R.uniform(0.5,0.8)))])
    return t
def aug_reorder_upi(t,R):
    p=t.split('/')
    if p[0]=='UPI' and len(p)>4: p[2],p[3]=p[3],p[2]
    return '/'.join(p)
def aug_ref_noise(t,R): return t+R.choice([' ',' /',' -'])+R.choice(['REF','UTR','TXN','RRN'])+r(R.randint(4,8))
def make_dialect(table):
    def f(t,R):
        for k,vs in table.items():
            if k in t: return t.replace(k,R.choice(vs))
        return t
    return f
AUGS_IN=[aug_delim,aug_case,aug_ocr,aug_trunc_tail,aug_drop_seg,aug_trunc_name,aug_reorder_upi,aug_ref_noise,make_dialect(DIALECT_IN)]
AUGS_OOD=AUGS_IN+[make_dialect(DIALECT_OOD)]
def label_consistent(t,lab_):
    u=t.upper(); errs=[]
    if lab_['is_emi'] and not any(c.upper()[:4] in u.replace(' ','') for c,_ in LEN): errs.append('emi_lender_cue')
    if lab_['is_bounce_or_return'] and not ('RTN' in u or 'RETURN' in u): errs.append('bounce_cue')
    if lab_['is_reversal'] and 'REV' not in u: errs.append('reversal_cue')
    cp=lab_['counterparty_name']
    if cp and cp.upper()[:4] not in u: errs.append('counterparty_cue')
    if lab_['p2p_or_p2m']=='P2M' and 'P2M' not in u: errs.append('p2m_cue')
    if lab_['sub_category']=='salary' and 'SAL' not in u: errs.append('salary_cue')
    return errs
def humanize(rows,augs,frac=HUMAN_FRAC,R=hrng):
    out=[];st={'rewritten':0,'dropped':0,'kept_clean':0}
    for x in rows:
        if R.random()>frac: out.append({**x,'humanized':False}); st['kept_clean']+=1; continue
        t=x['text']
        for a in R.sample(augs,R.randint(1,3)): t=a(t,R)
        t=t.strip()
        if t==x['text']: out.append({**x,'humanized':False}); st['kept_clean']+=1; continue
        if label_consistent(t,x['label']): out.append({**x,'humanized':False}); st['dropped']+=1
        else: out.append({'text':t,'label':x['label'],'humanized':True,'orig':x['text']}); st['rewritten']+=1
    return out,st
hsplits={}
for k in splits:
    hsplits[k],st=humanize(splits[k],AUGS_OOD if k=='ood' else AUGS_IN, frac=0.6 if k=='ood' else HUMAN_FRAC)
    with open(f'{DATA}/{k}_human.jsonl','w') as f: f.writelines(json.dumps(x)+'\n' for x in hsplits[k])
    print(k,st)
for x in [y for y in hsplits['train'] if y['humanized']][:4]: print(' ',x['orig'],'\n   ->',x['text'])
assert all(not label_consistent(x['text'],x['label']) for v in hsplits.values() for x in v if x['humanized'])
STATUS['A2_humanize']='PASS'

### A2 (optional, later) — Teacher-LLM rewrite stub
Not executed. Shows how a bigger model could paraphrase narrations into bank-specific dialects; outputs must go through the **same** `label_consistent` check before use.

In [ ]:
TEACHER_PROMPT = ("You rewrite Indian BANK STATEMENT narration lines (not SMS) into the style of a specific bank's statement.\n"
 "Keep every fact: amount-free, same counterparty, same lender, same EMI/bounce/reversal meaning.\n"
 "You may: abbreviate, truncate names, change delimiters, add ref numbers. Do NOT add new merchants or lenders.\n"
 "Bank style: {bank}\nLine: {text}\nReturn only the rewritten line.")
# def teacher_rewrite(text, bank='SBI'):
#     from openai import OpenAI                     # or any OpenAI-compatible endpoint / local vLLM server
#     cli = OpenAI(base_url=os.environ['TEACHER_URL'], api_key=os.environ['TEACHER_KEY'])
#     r = cli.chat.completions.create(model=os.environ['TEACHER_MODEL'], temperature=0.7,
#            messages=[{'role':'user','content':TEACHER_PROMPT.format(bank=bank,text=text)}])
#     return r.choices[0].message.content.strip()
# new = teacher_rewrite(x['text']); keep only if not label_consistent(new, x['label'])
print(TEACHER_PROMPT.format(bank='SBI',text=splits['train'][0]['text'])[:300])

## Component B — QLoRA training SMOKE (1 step)
**What:** load Qwen2.5-0.5B-Instruct in 4-bit (QLoRA), attach small LoRA adapters, run **1 SFT step**, save adapter.
**Why QLoRA:** base weights frozen and quantised to 4-bit → fits free T4; only ~1% params train.
**Instruct model:** already follows chat + emits JSON-ish output, so less data needed.
`USE_HUMANIZED=True` trains on `train_human.jsonl` (clean + rewritten mix).
**Backend:** if GPU and `USE_UNSLOTH=True` → try Unsloth `FastLanguageModel` (4-bit, max_seq_length=512, LoRA r=16); on any failure print the reason and fall back to BitsAndBytes + PEFT + TRL. Prints `TRAIN_BACKEND=unsloth|peft` (or `none` when skipped).
If no GPU: we still build the prompt formatting + LoRA config and print `SKIPPED_TRAIN=true`.

In [ ]:
MODEL_ID='Qwen/Qwen2.5-0.5B-Instruct'
SYS='Convert the bank statement narration into JSON with keys: '+', '.join(FIELDS)+'. Output JSON only.'
def to_msgs(ex): return {'messages':[{'role':'system','content':SYS},{'role':'user','content':ex['text']},
                                     {'role':'assistant','content':json.dumps(ex['label'])}]}
USE_HUMANIZED=True
train_rows=hsplits['train'] if USE_HUMANIZED else splits['train']
train_msgs=[to_msgs(x) for x in train_rows]; print('USE_HUMANIZED',USE_HUMANIZED,'humanized rows',sum(x.get('humanized',False) for x in train_rows)); print(train_msgs[0]['messages'][1])
SKIPPED_TRAIN=True; model=tok=None; ADAPTER=f'{BASE}/adapter_smoke'
try:
    from peft import LoraConfig
    lora_cfg=LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,task_type='CAUSAL_LM',
                        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
    print('LoRA config OK')
except ImportError: print('peft not installed (local dry-run) -> config skipped')
TRAIN_BACKEND='none'
def _sft_cfg():
    from trl import SFTConfig
    return SFTConfig(output_dir=f'{BASE}/out',max_steps=1,per_device_train_batch_size=2,learning_rate=2e-4,
                     logging_steps=1,report_to=[],fp16=True)
if HAS_GPU and USE_UNSLOTH:
    try:
        from unsloth import FastLanguageModel
        from trl import SFTTrainer
        from datasets import Dataset
        model,tok=FastLanguageModel.from_pretrained(model_name=MODEL_ID,max_seq_length=512,load_in_4bit=True)
        model=FastLanguageModel.get_peft_model(model,r=16,lora_alpha=32,lora_dropout=0,
                    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
        tr=SFTTrainer(model=model,args=_sft_cfg(),train_dataset=Dataset.from_list(train_msgs),processing_class=tok)
        tr.train(); tr.save_model(ADAPTER); model=tr.model; SKIPPED_TRAIN=False; TRAIN_BACKEND='unsloth'
    except Exception as e:
        print('Unsloth path failed -> falling back to PEFT+TRL:',repr(e)[:300]); model=tok=None
if HAS_GPU and SKIPPED_TRAIN:
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        from trl import SFTTrainer
        from datasets import Dataset
        tok=AutoTokenizer.from_pretrained(MODEL_ID)
        bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16)
        model=AutoModelForCausalLM.from_pretrained(MODEL_ID,quantization_config=bnb,device_map='auto')
        tr=SFTTrainer(model=model,args=_sft_cfg(),train_dataset=Dataset.from_list(train_msgs),peft_config=lora_cfg,processing_class=tok)
        tr.train(); tr.save_model(ADAPTER); model=tr.model; SKIPPED_TRAIN=False; TRAIN_BACKEND='peft'
    except Exception as e: print('Train failed/OOM:',repr(e)[:300]); model=tok=None
print('TRAIN_BACKEND='+TRAIN_BACKEND)
print('SKIPPED_TRAIN=',str(SKIPPED_TRAIN).lower()); STATUS['B_train']='SKIPPED' if SKIPPED_TRAIN else 'PASS (1 step, '+TRAIN_BACKEND+')'

## Component B2 — Optional DPO smoke (preference tuning, 1 step)
**What:** tiny preference set: `chosen` = gold JSON; `rejected` = typical failure modes — invalid JSON, wrong `is_emi`, swapped category, hallucinated merchant/counterparty.
**Why:** DPO here is mainly for **schema / format discipline** (stop broken JSON and obvious flips). Some public repos (e.g. the QLoRA+DPO Qwen2.5-0.5B JSON-extraction recipes in our research doc) report DPO helping schema pass, while others note it can hurt field recall — **the team should measure on our own OOD set**; no numbers are claimed here.
Runs TRL `DPOTrainer` max_steps=1 on top of the SFT adapter when GPU+peft exist; else builds the dataset and prints `SKIPPED_DPO=true`.

In [ ]:
prng=random.Random(5)
CATS=sorted({x['label']['category'] for x in splits['train']})
def rej_invalid(l): return json.dumps(l)[:-prng.randint(2,15)]
def rej_emi(l): return json.dumps({**l,'is_emi':not l['is_emi']})
def rej_cat(l): return json.dumps({**l,'category':prng.choice([c for c in CATS if c!=l['category']])})
def rej_halluc(l): return json.dumps({**l,'counterparty_name':prng.choice(['PAYTM MALL','MYNTRA','RAJESH GUPTA']),'counterparty_type':'merchant'})
REJ=[rej_invalid,rej_emi,rej_cat,rej_halluc]
def prompt_of(t): return [{'role':'system','content':SYS},{'role':'user','content':t}]
pref=[]
for x in train_rows[:40]:
    f=prng.choice(REJ)
    pref.append({'prompt':prompt_of(x['text']),'chosen':[{'role':'assistant','content':json.dumps(x['label'])}],
                 'rejected':[{'role':'assistant','content':f(x['label'])}],'failure':f.__name__})
with open(f'{DATA}/dpo_pref.jsonl','w') as fo: fo.writelines(json.dumps(p)+'\n' for p in pref)
from collections import Counter; print('pref pairs',len(pref),Counter(p['failure'] for p in pref))
assert all(p['chosen'][0]['content']!=p['rejected'][0]['content'] for p in pref)
SKIPPED_DPO=True
if HAS_GPU and model is not None:
    try:
        from trl import DPOTrainer, DPOConfig
        from datasets import Dataset
        dcfg=DPOConfig(output_dir=f'{BASE}/dpo_out',max_steps=1,per_device_train_batch_size=2,learning_rate=5e-6,
                       beta=0.1,logging_steps=1,report_to=[],fp16=True,max_length=512)
        dtr=DPOTrainer(model=model,args=dcfg,train_dataset=Dataset.from_list([{k:p[k] for k in ('prompt','chosen','rejected')} for p in pref]),processing_class=tok)
        dtr.train(); dtr.save_model(f'{BASE}/adapter_dpo_smoke'); model=dtr.model; SKIPPED_DPO=False
    except Exception as e: print('DPO failed:',repr(e)[:300])
print('SKIPPED_DPO=',str(SKIPPED_DPO).lower()); STATUS['B2_dpo']='SKIPPED (dataset built)' if SKIPPED_DPO else 'PASS (1 step)'

## Component D (defined early) — `tag_line(text) -> dict`
Order: trained/loaded model → generate JSON; else deterministic **regex fallback** so the demo always returns JSON.
(Defined before eval so Component C can score it.)

In [ ]:
def rule_tag(t):
    u=t.upper(); uc=u.replace(' ',''); base=lab('other','unknown',None,'unknown'); base['confidence']=0.3
    lend=next((n for c,n in LEN if c.replace(' ','')[:4] in uc),None)
    seg=lambda names:next((n for n in names if n[:4] in u),None)
    if 'RTN' in u or 'RETURN' in u: d=lab('charges','emi_bounce_charge',lend,'lender',emi=True,ln=lend,bnc=True)
    elif 'REV' in u[:6]: d=lab('refund','imps_reversal',None,'bank',rev=True)
    elif lend and any(k in u for k in ('NACH','ECS','EMI','MANDATE','ACH')): d=lab('loan','emi_payment',lend,'lender',emi=True,ln=lend)
    elif 'ATM' in u: d=lab('cash','atm_withdrawal',None,'bank')
    elif 'SAL' in u: d=lab('income','salary',seg(EMP),'employer')
    elif 'P2M' in u: d=lab('expense','merchant_payment',seg(MER),'merchant',pp='P2M')
    elif 'P2A' in u or 'IMPS' in u or seg(PPL): d=lab('transfer','p2p_transfer',seg(PPL),'individual',pp='P2P')
    else: return base
    d['confidence']=0.8; return d
def model_tag(t):
    import torch
    p=tok.apply_chat_template([{'role':'system','content':SYS},{'role':'user','content':t}],tokenize=False,add_generation_prompt=True)
    ids=tok(p,return_tensors='pt').to(model.device)
    with torch.no_grad(): o=model.generate(**ids,max_new_tokens=200,do_sample=False)
    return tok.decode(o[0][ids['input_ids'].shape[1]:],skip_special_tokens=True)
def tag_line(t, raw=False):
    if model is not None:
        s=model_tag(t)
        if raw: return s
        try: return json.loads(s[s.find('{'):s.rfind('}')+1])
        except Exception: pass
    return json.dumps(rule_tag(t)) if raw else rule_tag(t)
TAG_MODE='model' if model is not None else 'regex_fallback'; print('tag mode:',TAG_MODE)

## Component C — Eval smoke (10 lines)
Splits scored: `val`, `ood` (clean held-out templates) and **`ood_human`** (held-out templates + humanized/unseen dialects — the preferred headline split).
Metrics: **valid JSON %**, **schema pass %** (jsonschema), **per-field exact match** (excluding `confidence`).
On a 1-step model these numbers are meaningless — the point is that the harness runs.

In [ ]:
V=jsonschema.Draft7Validator(SCHEMA)
def evaluate(rows):
    vj=sp=0; fm={f:0 for f in FIELDS if f!='confidence'}
    for x in rows:
        try: p=json.loads(tag_line(x['text'],raw=True)); vj+=1
        except Exception: continue
        sp+=V.is_valid(p)
        for f in fm: fm[f]+= p.get(f)==x['label'][f]
    n=len(rows); return {'valid_json%':100*vj/n,'schema_pass%':100*sp/n,**{f:100*c/n for f,c in fm.items()}}
EV={'val':splits['val'][:10],'ood':splits['ood'][:10],'ood_human':hsplits['ood'][:15]}
res={k:evaluate(v) for k,v in EV.items()}
print(f"{'metric':22}"+''.join(f"{k:>11}" for k in res))
for m in res['val']: print(f"{m:22}"+''.join(f"{res[k][m]:11.1f}" for k in res))
json.dump({'mode':TAG_MODE,**res},open(f'{BASE}/eval_results.json','w'),indent=1); STATUS['C_eval']='PASS ('+TAG_MODE+')'

## Component D — Inference demo

In [ ]:
demo=['UPI/P2M/412398765432/SWIGGY/PAYMENT FROM PHONE/YESB','NACH-DR-BAJAJFIN-0099887766',
      'ACH RTN CHG TATACAP 55443322','REV-IMPS-998877665544-FAILED TXN','ATM WDL/123456/PUNE','upi p2m 4123987 swig paymen','ECS DR BAJAJFIN 77881 REF8812','NEFT CR-HDFC0000123-TCS LTD-SALARY MAR']
out=[{'text':t,'json':tag_line(t)} for t in demo]
for o in out: print(o['text'],'\n  ->',json.dumps(o['json']))
json.dump(out,open(f'{BASE}/infer_demo.json','w'),indent=1); STATUS['D_infer']='PASS ('+TAG_MODE+')'

## Component E — Serving with vLLM (real run on Colab GPU)
**What:** `USE_VLLM=True` (default) → on Colab + GPU: `pip install vllm`, merge the smoke LoRA adapter into the fp16 base (or use the **base model** if training was skipped), load it with vLLM, and run narration → JSON for the demo lines.
**Why vLLM:** continuous batching + paged KV cache = high throughput for many short JSON generations; same engine used later for the team GPU demo (`vllm serve` → OpenAI-compatible API).
**Fallbacks:** if install / load / generate fails → Transformers `generate` (if a model is loaded) → regex. Prints `INFER_BACKEND=vllm|transformers|regex`.
`SKIP_GPU=1` or no GPU → skipped cleanly, **no vLLM install**.
Note: installing vLLM on Colab can upgrade torch; if later cells misbehave, *Runtime → Restart session* and re-run. Run this cell last.

In [ ]:
USE_VLLM = True   # flip to False to skip vLLM
INFER_BACKEND = 'transformers' if model is not None else 'regex'; vllm_out=None
if IN_COLAB and HAS_GPU and USE_VLLM:
    try:
        !pip -q install vllm
        import torch, gc
        from transformers import AutoTokenizer, AutoModelForCausalLM
        SERVE_DIR = MODEL_ID
        if not SKIPPED_TRAIN:
            from peft import PeftModel
            last = f'{BASE}/adapter_dpo_smoke' if not SKIPPED_DPO else ADAPTER
            basem = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
            merged = PeftModel.from_pretrained(basem, last).merge_and_unload()
            SERVE_DIR = f'{BASE}/merged_model'; merged.save_pretrained(SERVE_DIR)
            AutoTokenizer.from_pretrained(MODEL_ID).save_pretrained(SERVE_DIR); del basem, merged
        gc.collect(); torch.cuda.empty_cache()
        from vllm import LLM, SamplingParams
        llm = LLM(model=SERVE_DIR, dtype='float16', max_model_len=512, gpu_memory_utilization=0.5, enforce_eager=True)
        sp = SamplingParams(temperature=0, max_tokens=200)
        outs = llm.chat([[{'role':'system','content':SYS},{'role':'user','content':t}] for t in demo], sp)
        vllm_out = []
        for t,o in zip(demo,outs):
            s=o.outputs[0].text
            try: j=json.loads(s[s.find('{'):s.rfind('}')+1]); ok=True
            except Exception: j=s; ok=False
            vllm_out.append({'text':t,'json':j,'valid_json':ok}); print(t,'\n  ->',s[:200])
        print('vLLM valid JSON:',sum(x['valid_json'] for x in vllm_out),'/',len(vllm_out),'| served from',SERVE_DIR)
        json.dump(vllm_out,open(f'{BASE}/infer_vllm.json','w'),indent=1); INFER_BACKEND='vllm'
    except Exception as e:
        print('vLLM failed -> fallback (transformers/regex) used in Component D:', repr(e)[:300])
else:
    print('vLLM skipped (needs Colab + GPU + USE_VLLM=True)')
print('INFER_BACKEND='+INFER_BACKEND)
STATUS['E_vllm']='PASS (vllm, '+('base' if SKIPPED_TRAIN else 'merged adapter')+')' if INFER_BACKEND=='vllm' else 'SKIPPED -> '+INFER_BACKEND

## Summary — what ran vs skipped

In [ ]:
for k,v in STATUS.items(): print(f'{k:10} {v}')
json.dump(STATUS,open(f'{BASE}/status.json','w'),indent=1)